##### Import the required modules and configure the system path to locate them

In [1]:
import sys
sys.path.append("../../utils")
import networkx
import astra_sim_sdk.astra_sim_sdk as astra_sim_kit
from astra_sim import AstraSim, Collective, NetworkBackend
from infragraph.infragraph_service import InfraGraphService
from infragraph.blueprints.devices.ironwood_rack import IronwoodRack
from infragraph import Infrastructure

##### Call the AstraSim client helper with the server endpoint and tag to connect to the ASTRA-sim gRPC server, initialize the SDK, and create a tagged folder for configs, results, and logs

In [2]:
astra = AstraSim(server_endpoint = "172.17.0.2:8989", tag = "analytical_ironwood_rack")

Resetting test directory
All contents of the folder /workspaces/astra_sim_service/client-scripts/utils/../trial/analytical_ironwood_rack have been deleted.
Successfully connected to gRPC server at 172.17.0.2:8989


##### Create a ironwood rack device fabric using infragraph device blueprint

In [3]:
server = IronwoodRack()
infrastructure = Infrastructure()
infrastructure.devices.append(server)

infrastructure.instances.add(name=server.name, device=server.name, count=1)
astra.configuration.infragraph.infrastructure.deserialize(infrastructure.serialize())
print(astra.configuration.infragraph.infrastructure)

devices:
- components:
  - choice: xpu
    count: 64
    description: Ironwood TPU
    name: IRONWOOD
  - choice: cpu
    count: 32
    description: AMD EPYC CPU
    name: AMD_TURIN_ZEN5_EPYC
  - choice: nic
    count: 32
    description: Network Interface Card
    name: nic200
  description: Rack with 64 XPUs in 4x4x4 Torus, 32 CPUs, and 32 NICs
  edges:
  - ep1:
      component: IRONWOOD[0]
    ep2:
      component: IRONWOOD[1]
    link: ici
    scheme: one2one
  - ep1:
      component: IRONWOOD[0]
    ep2:
      component: IRONWOOD[4]
    link: ici
    scheme: one2one
  - ep1:
      component: IRONWOOD[0]
    ep2:
      component: IRONWOOD[16]
    link: ici
    scheme: one2one
  - ep1:
      component: IRONWOOD[1]
    ep2:
      component: IRONWOOD[2]
    link: ici
    scheme: one2one
  - ep1:
      component: IRONWOOD[1]
    ep2:
      component: IRONWOOD[5]
    link: ici
    scheme: one2one
  - ep1:
      component: IRONWOOD[1]
    ep2:
      component: IRONWOOD[17]
    link: ici


##### Initialize the Infragraph service, display the fabric topology, and retrieve/set the total number of NPUs to generate the collective

In [4]:
service = InfraGraphService()
service.set_graph(infrastructure)
total_npus = service.get_component(device=server, type="xpu").count
g = service.get_networkx_graph()
print(networkx.write_network_text(g, vertical_chains=True))

╙── IRONWOOD_TORUS_RACK_4x4x4.0.nic200.17
    │
    IRONWOOD_TORUS_RACK_4x4x4.0.AMD_TURIN_ZEN5_EPYC.17
    ├── IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.34
    │   ├── IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.18
    │   │   ├── IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.2
    │   │   │   ├── IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.1
    │   │   │   │   ├── IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.0
    │   │   │   │   │   ├── IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.4
    │   │   │   │   │   │   ├── IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.5 ─ IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.1
    │   │   │   │   │   │   │   ├── IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.6 ─ IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.2
    │   │   │   │   │   │   │   │   ├── IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.7 ─ IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.4
    │   │   │   │   │   │   │   │   │   ├── IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.3 ─ IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.2, IRONWOOD_TORUS_RACK_4x4x4.0.IRONWOOD.0
    │   │   │   │   │   │   │   │   │   │

##### Generate workload execution traces for each rank and set the required data size for AstraSim configuration

In [5]:
astra.configuration.common_config.workload = astra.generate_collective(collective=Collective.ALLREDUCE, coll_size= 1 *1024*1024, npu_range=[0, total_npus])

Generated 64 et in /workspaces/astra_sim_service/client-scripts/utils/../trial/analytical_ironwood_rack/configuration/workload


##### Configure ASTRA-sim system config

In [6]:
astra.configuration.common_config.system.scheduling_policy = astra.configuration.common_config.system.LIFO
astra.configuration.common_config.system.endpoint_delay = 10
astra.configuration.common_config.system.active_chunks_per_dimension = 1
astra.configuration.common_config.system.all_gather_implementation = [astra.configuration.common_config.system.RING, astra.configuration.common_config.system.RING, astra.configuration.common_config.system.RING]
astra.configuration.common_config.system.all_to_all_implementation = [astra.configuration.common_config.system.RING, astra.configuration.common_config.system.RING, astra.configuration.common_config.system.RING]
astra.configuration.common_config.system.all_reduce_implementation = [astra.configuration.common_config.system.RING, astra.configuration.common_config.system.RING, astra.configuration.common_config.system.RING]
astra.configuration.common_config.system.collective_optimization = astra.configuration.common_config.system.LOCALBWAWARE
astra.configuration.common_config.system.local_mem_bw = 1600

##### Configure ASTRA-sim remote memory configuration

In [7]:
astra.configuration.common_config.remote_memory.memory_type = astra.configuration.common_config.remote_memory.NO_MEMORY_EXPANSION
print(astra.configuration.common_config.remote_memory)

memory_type: NO_MEMORY_EXPANSION
remote_mem_bw: 0
remote_mem_latency: 0



##### Configure network backend to ANALYTICAL_CONGESTION_AWARE

In [8]:
astra.configuration.network_backend.analytical_congestion_unaware.topology.choice = astra.configuration.network_backend.analytical_congestion_unaware.topology.INFRAGRAPH

##### Adding ASTRA-sim - Infragraph specific annotation

In [9]:
host_device_spec = astra_sim_kit.AnnotationDeviceSpecifications()
host_device_spec.device_bandwidth_gbps = 100
host_device_spec.device_latency_ms = 0.05
host_device_spec.device_name = server.name
host_device_spec.device_type = "host"
astra.configuration.infragraph.annotations.device_specifications.append(host_device_spec)


##### Configure ASTRA-sim cmd parameters

In [10]:
astra.configuration.common_config.cmd_parameters.comm_scale = 1
astra.configuration.common_config.cmd_parameters.injection_scale = 1
astra.configuration.common_config.cmd_parameters.rendezvous_protocol = False

#### Start the simulation by specifying the network backend

In [11]:
astra.run_simulation(NetworkBackend.ANALYTICAL_CONGESTION_UNAWARE)

Generating Configuration ZIP now
output_path: /workspaces/astra_sim_service/client-scripts/utils/../trial/analytical_ironwood_rack/config.zip
folder_path: /workspaces/astra_sim_service/client-scripts/utils/../trial/analytical_ironwood_rack/configuration/workload/..
pack_zip complete
message: 'Configuration applied successfully. warnings: Unable to generate communicator
  group message from schema - communicator group configuration empty'

message: Simulation started successfully

astra-sim server Status: running
astra-sim server Status: running
Transferring Files from ASTRA-sim server
All files downloaded Successfully
Simulation completed


##### Download all the configurations as a zip

In [12]:
astra.download_configuration()

Downloaded all configuration in /workspaces/astra_sim_service/client-scripts/utils/../trial/analytical_ironwood_rack/server_configuration.zip


##### Save infragraph as a yaml

In [13]:
import yaml
import os
from common import FileFolderUtils
with open(os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR,"../infrastructure","analytical_dgx_device.yaml"),"w") as f:
    data = infrastructure.serialize("dict")
    yaml.dump(data, f, default_flow_style=False, indent=4)

print("saved yaml to:", os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR,"..","analytical_dgx_device.yaml"))

saved yaml to: /workspaces/astra_sim_service/client-scripts/utils/../trial/analytical_ironwood_rack/output/../analytical_dgx_device.yaml
